In [ ]:
# !pip install -q pandas numpy scikit-learn matplotlib seaborn tensorflow joblib tf2onnx xgboost

In [ ]:
import os, json, math, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (classification_report, f1_score,
                             accuracy_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# from google.colab import files   # uncomment on Colab

In [ ]:
# uploaded = files.upload()

FILE_NAME = "audit_trail_2025.csv"
df = pd.read_csv(FILE_NAME)
print(f"Loaded {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
df["createdAt"] = pd.to_datetime(df["createdAt"], utc=True, errors="coerce")

list_cols = ["companyIdList","companyGroupIdList","insurerIdList",
             "companySectionIdList","insurerCodeIdList",
             "healthcareNetworkIdList","domainIdList"]
for c in list_cols:
    df[c] = df[c].fillna("[]").astype(str)

text_cols = ["action","type","subType","device","countryCode","city","status","userAgent","anomaly_type"]
for c in text_cols:
    if c in df.columns:
        df[c] = df[c].fillna("UNKNOWN").astype(str)

if "is_anomaly" not in df.columns:
    df["is_anomaly"] = 0
if "anomaly_type" not in df.columns:
    df["anomaly_type"] = "normal"
if "sessionLength" not in df.columns:
    df["sessionLength"] = df.groupby(["insuredId","sessionNumber"])["sequenceInSession"].transform("max")

df = df.sort_values(["insuredId","sessionNumber","sequenceInSession"]).reset_index(drop=True)
print("Normal rows  :", (df["is_anomaly"] == 0).sum())
print("Anomaly rows :", (df["is_anomaly"] == 1).sum())
if df["is_anomaly"].sum() > 0:
    print("Anomaly types:\n", df[df["is_anomaly"]==1]["anomaly_type"].value_counts())

In [ ]:
df["hour"]      = df["createdAt"].dt.hour
df["dayofweek"] = df["createdAt"].dt.dayofweek
df["month_num"] = df["createdAt"].dt.month
df["hour_sin"]  = np.sin(2*np.pi*df["hour"]/24)
df["hour_cos"]  = np.cos(2*np.pi*df["hour"]/24)
df["dow_sin"]   = np.sin(2*np.pi*df["dayofweek"]/7)
df["dow_cos"]   = np.cos(2*np.pi*df["dayofweek"]/7)
df["is_ok"] = (df["status"]=="OK").astype(int)
df["is_ko"] = (df["status"]=="KO").astype(int)

In [ ]:
action_le  = LabelEncoder()
device_le  = LabelEncoder()
country_le = LabelEncoder()
type_le    = LabelEncoder()
subtype_le = LabelEncoder()

df["action_id"]       = action_le.fit_transform(df["action"])
df["device_id"]       = device_le.fit_transform(df["device"])
df["country_id"]      = country_le.fit_transform(df["countryCode"])
df["type_id"]         = type_le.fit_transform(df["type"])
df["subType_filled"]  = df["subType"].fillna("NONE").astype(str)
df["subtype_id"]      = subtype_le.fit_transform(df["subType_filled"])

print("Unique actions:", len(action_le.classes_))

In [ ]:
SESSION_KEY = ["insuredId","sessionNumber"]
df["prev_time"]  = df.groupby(SESSION_KEY)["createdAt"].shift(1)
df["delta_seconds"] = (df["createdAt"]-df["prev_time"]).dt.total_seconds().fillna(0)
df["delta_seconds_clipped"] = df["delta_seconds"].clip(0, 3600)

scaler_delta = StandardScaler()
df["delta_scaled"] = scaler_delta.fit_transform(df[["delta_seconds_clipped"]])

In [ ]:
df["seq_pos_norm"]     = df["sequenceInSession"] / df["sessionLength"]
df["is_session_start"] = (df["sequenceInSession"]==1).astype(int)
df["is_session_end"]   = (df["sequenceInSession"]==df["sessionLength"]).astype(int)
df["prev_action_id"]   = (df.groupby(SESSION_KEY)["action_id"].shift(1).fillna(-1).astype(int)+1)
df["ko_count_so_far"]  = (df.groupby(SESSION_KEY)["is_ko"].cumsum().shift(1).fillna(0).astype(int))
max_session_len        = df["sessionLength"].max()
df["session_len_norm"] = df["sessionLength"] / max_session_len

# ★ NEW: ip_changed_flag (within session) — key geo_jump signal
df["ip_prev"] = df.groupby(SESSION_KEY)["ip"].shift(1)
df["ip_changed"] = ((df["ip"]!=df["ip_prev"]) & df["ip_prev"].notna()).astype(int)

print("Session features ready.")

In [ ]:
SEQ_LEN = 10

feature_cols = [
    # Categorical
    "action_id","device_id","country_id","type_id","subtype_id",
    "prev_action_id",
    # Temporal
    "hour_sin","hour_cos","dow_sin","dow_cos","month_num",
    # Time gap
    "delta_scaled",
    # Status
    "is_ok","is_ko",
    # Session context
    "seq_pos_norm","is_session_start","is_session_end","session_len_norm",
    "ko_count_so_far",
    # ★ NEW
    "ip_changed",
]
print(f"Feature vector size: {len(feature_cols)}")

In [ ]:
def build_sequences(df, seq_len=10, include_anomaly_label=False):
    X, y_next, y_anomaly, y_atype, meta = [], [], [], [], []
    atype_le_local = None
    if include_anomaly_label:
        atype_le_local = LabelEncoder()
        df["anomaly_type_id"] = atype_le_local.fit_transform(df["anomaly_type"].fillna("normal"))

    for (uid, snum), g in df.groupby(SESSION_KEY):
        g = g.sort_values("sequenceInSession")
        vals    = g[feature_cols].values.astype(np.float32)
        actions = g["action_id"].values
        times   = g["createdAt"].values
        # Session-level anomaly label (1 if ANY event is anomalous)
        sess_anom = int(g["is_anomaly"].max()) if "is_anomaly" in g else 0
        sess_type = g["anomaly_type"].iloc[0] if "anomaly_type" in g.columns else "normal"

        n = len(g)
        if n < 2:
            continue
        for end in range(1, n):
            start = max(0, end - seq_len)
            seq   = vals[start:end]
            if len(seq) < seq_len:
                pad = np.zeros((seq_len-len(seq), len(feature_cols)), dtype=np.float32)
                seq = np.vstack([pad, seq])
            X.append(seq)
            y_next.append(actions[end])
            y_anomaly.append(sess_anom)
            y_atype.append(sess_type)
            meta.append({"insuredId": uid, "sessionNumber": snum,
                         "end_time": str(times[end-1]), "next_time": str(times[end])})

    return (np.array(X, dtype=np.float32),
            np.array(y_next, dtype=np.int32),
            np.array(y_anomaly, dtype=np.int32),
            y_atype, meta, atype_le_local)

X_all, y_all, y_anom_all, y_atype_all, meta_all, _ = build_sequences(df, SEQ_LEN, include_anomaly_label=False)
print("X_all:", X_all.shape, "| y_all:", y_all.shape)

In [ ]:
meta_df  = pd.DataFrame(meta_all)
meta_df["end_time"] = pd.to_datetime(meta_df["end_time"], utc=True)
order    = np.argsort(meta_df["end_time"].values)
X_all    = X_all[order];   y_all = y_all[order]
y_anom_all  = y_anom_all[order]
y_atype_all = [y_atype_all[i] for i in order]
meta_df  = meta_df.iloc[order].reset_index(drop=True)

split    = int(len(X_all)*0.8)
X_train, X_test = X_all[:split], X_all[split:]
y_train, y_test = y_all[:split], y_all[split:]
y_anom_train, y_anom_test = y_anom_all[:split], y_anom_all[split:]
y_atype_train = y_atype_all[:split];  y_atype_test = y_atype_all[split:]

print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print(f"Anomaly in train: {y_anom_train.sum()}  Anomaly in test: {y_anom_test.sum()}")

In [ ]:
artifacts_dir = "artifacts"
os.makedirs(artifacts_dir, exist_ok=True)

for name, le in [("action",action_le),("device",device_le),
                 ("country",country_le),("type",type_le),("subtype",subtype_le)]:
    with open(f"{artifacts_dir}/{name}_vocab.json","w",encoding="utf-8") as f:
        json.dump({int(i): cls for i,cls in enumerate(le.classes_)}, f, ensure_ascii=False, indent=2)

joblib.dump(scaler_delta, f"{artifacts_dir}/scaler_delta.pkl")
json.dump({
    "seq_len": SEQ_LEN, "feature_cols": feature_cols,
    "n_features": len(feature_cols),
    "action_vocab_size": int(len(action_le.classes_)),
    "device_vocab_size": int(len(device_le.classes_)),
    "country_vocab_size": int(len(country_le.classes_)),
    "type_vocab_size": int(len(type_le.classes_)),
    "subtype_vocab_size": int(len(subtype_le.classes_)),
    "max_session_len": int(max_session_len),
}, open(f"{artifacts_dir}/feature_config.json","w"), ensure_ascii=False, indent=2)

print("Artefacts saved.")

#  ██████╗  █████╗ ██████╗ ████████╗     █████╗
#  ██╔══██╗██╔══██╗██╔══██╗╚══██╔══╝    ██╔══██╗
#  ██████╔╝███████║██████╔╝   ██║       ███████║
#  ██╔═══╝ ██╔══██║██╔══██╗   ██║       ██╔══██║
#  ██║     ██║  ██║██║  ██║   ██║       ██║  ██║
#  ╚═╝     ╚═╝  ╚═╝╚═╝  ╚═╝   ╚═╝       ╚═╝  ╚═╝
#  LSTM AUTOENCODER — Anomaly DETECTION (binary)

In [ ]:
# Train ONLY on sequences with no anomaly → model learns "normal"
normal_train_mask = (y_anom_train == 0)
X_train_ae = X_train[normal_train_mask]

# For evaluation: split test into normal vs anomaly subsets
normal_test_mask  = (y_anom_test == 0)
anomaly_test_mask = (y_anom_test == 1)
X_test_normal  = X_test[normal_test_mask]
X_test_anomaly = X_test[anomaly_test_mask]

print(f"AE train (normal only): {X_train_ae.shape}")
print(f"AE test  normal  : {X_test_normal.shape}")
print(f"AE test  anomaly : {X_test_anomaly.shape}")

In [ ]:
input_dim = X_train.shape[-1]

ae_inputs = layers.Input(shape=(SEQ_LEN, input_dim))
x = layers.Masking(mask_value=0.0)(ae_inputs)
# Encoder — Bidirectional LSTM
x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
x = layers.Dropout(0.20)(x)
x = layers.LSTM(32, return_sequences=False)(x)
# Bottleneck
x = layers.RepeatVector(SEQ_LEN)(x)
# Decoder
x = layers.LSTM(32, return_sequences=True)(x)
x = layers.Dropout(0.20)(x)
x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
ae_outputs = layers.TimeDistributed(layers.Dense(input_dim))(x)

autoencoder = models.Model(ae_inputs, ae_outputs, name="lstm_autoencoder")
autoencoder.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
autoencoder.summary()

In [ ]:
X_tr_ae, X_val_ae = train_test_split(X_train_ae, test_size=0.15, shuffle=False)

es_ae = callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
lr_ae = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6)

history_ae = autoencoder.fit(
    X_tr_ae, X_tr_ae,
    validation_data=(X_val_ae, X_val_ae),
    epochs=60, batch_size=128,
    callbacks=[es_ae, lr_ae], verbose=1,
)

plt.figure(figsize=(8,4))
plt.plot(history_ae.history["loss"], label="train")
plt.plot(history_ae.history["val_loss"], label="val")
plt.title("LSTM Autoencoder — Reconstruction Loss"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
val_pred   = autoencoder.predict(X_val_ae, verbose=0)
val_errors = np.mean(np.square(X_val_ae - val_pred), axis=(1,2))

Q1, Q3 = np.percentile(val_errors, [25, 75])
IQR     = Q3 - Q1
threshold = float(Q3 + 1.5 * IQR)   # standard Tukey fence
print(f"Threshold (IQR) : {threshold:.4f}")
print(f"Threshold (p95) : {np.percentile(val_errors, 95):.4f}")

with open(f"{artifacts_dir}/anomaly_threshold.json","w") as f:
    json.dump({"threshold": threshold, "Q1": float(Q1), "Q3": float(Q3), "IQR": float(IQR)}, f, indent=2)

plt.figure(figsize=(8,4))
sns.histplot(val_errors, bins=60, kde=True)
plt.axvline(threshold, color="red", linestyle="--", label=f"IQR threshold={threshold:.3f}")
plt.legend(); plt.title("Validation reconstruction error (normal sessions only)"); plt.show()

In [ ]:
err_normal  = np.mean(np.square(X_test_normal  - autoencoder.predict(X_test_normal,  verbose=0)), axis=(1,2))
err_anomaly = np.mean(np.square(X_test_anomaly - autoencoder.predict(X_test_anomaly, verbose=0)), axis=(1,2))

all_errors = np.concatenate([err_normal, err_anomaly])
all_labels = np.array([0]*len(err_normal) + [1]*len(err_anomaly))

flags = (all_errors > threshold).astype(int)
tp = ((flags==1) & (all_labels==1)).sum()
fp = ((flags==1) & (all_labels==0)).sum()
fn = ((flags==0) & (all_labels==1)).sum()
precision = tp / max(1, tp+fp)
recall    = tp / max(1, tp+fn)
f1        = 2*precision*recall / max(1e-9, precision+recall)
auc       = roc_auc_score(all_labels, all_errors)

print(f"AUC-ROC   : {auc:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1        : {f1:.4f}")

In [ ]:
autoencoder.export(f"{artifacts_dir}/anomaly_lstm_autoencoder")
# !python -m tf2onnx.convert --saved-model artifacts/anomaly_lstm_autoencoder \
#   --output artifacts/anomaly_lstm_autoencoder.onnx

#  ██████╗  █████╗ ██████╗ ████████╗     ██████╗
#  ██╔══██╗██╔══██╗██╔══██╗╚══██╔══╝    ██╔══██╗
#  ██████╔╝███████║██████╔╝   ██║       ██████╔╝
#  ██╔═══╝ ██╔══██║██╔══██╗   ██║       ██╔══██╗
#  ██║     ██║  ██║██║  ██║   ██║       ██████╔╝
#  ╚═╝     ╚═╝  ╚═╝╚═╝  ╚═╝   ╚═╝       ╚═════╝
#  GRU — Next-Action Prediction

In [ ]:
num_actions = len(action_le.classes_)
classes_present = np.unique(y_train)
raw_weights = compute_class_weight("balanced", classes=classes_present, y=y_train)
raw_weights = np.clip(raw_weights, 0.1, 10.0)
class_weight_dict = {int(c): float(w) for c,w in zip(classes_present, raw_weights)}
for c in range(num_actions):
    class_weight_dict.setdefault(c, 1.0)
print(f"Class weights computed for {len(class_weight_dict)} classes.")

In [ ]:
gru_in = layers.Input(shape=(SEQ_LEN, input_dim))
gx = layers.Masking(mask_value=0.0)(gru_in)
gx = layers.Bidirectional(layers.GRU(128, return_sequences=True))(gx)
gx = layers.Dropout(0.30)(gx)
gx = layers.Bidirectional(layers.GRU(64, return_sequences=False))(gx)
gx = layers.Dropout(0.30)(gx)
gx = layers.Dense(128, activation="relu")(gx)
gx = layers.Dropout(0.20)(gx)
gru_out = layers.Dense(num_actions, activation="softmax")(gx)

gru_model = models.Model(gru_in, gru_out, name="gru_next_action")
gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
gru_model.summary()

In [ ]:
X_tr_gru, X_val_gru, y_tr_gru, y_val_gru = train_test_split(
    X_train, y_train, test_size=0.15, shuffle=False)

es_gru = callbacks.EarlyStopping(monitor="val_accuracy", patience=12,
                                  restore_best_weights=True, mode="max")
lr_gru = callbacks.ReduceLROnPlateau(monitor="val_accuracy", factor=0.5,
                                      patience=4, min_lr=1e-6, mode="max")

history_gru = gru_model.fit(
    X_tr_gru, y_tr_gru,
    validation_data=(X_val_gru, y_val_gru),
    epochs=60, batch_size=128,
    class_weight=class_weight_dict,
    callbacks=[es_gru, lr_gru], verbose=1,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(history_gru.history["accuracy"], label="train")
ax1.plot(history_gru.history["val_accuracy"], label="val")
ax1.set_title("GRU Accuracy"); ax1.legend()
ax2.plot(history_gru.history["loss"], label="train")
ax2.plot(history_gru.history["val_loss"], label="val")
ax2.set_title("GRU Loss"); ax2.legend()
plt.tight_layout(); plt.show()

In [ ]:
y_prob = gru_model.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred, average="macro", zero_division=0)
print(f"Top-1 Accuracy : {acc:.4f}")
print(f"Macro-F1       : {f1:.4f}")

def top_k_accuracy(y_true, y_prob, k):
    topk = np.argsort(y_prob, axis=1)[:, -k:]
    return np.mean([1 if y_true[i] in topk[i] else 0 for i in range(len(y_true))])

for k in [1, 3, 5]:
    print(f"Top-{k} accuracy: {top_k_accuracy(y_test, y_prob, k):.4f}")

print(classification_report(y_test, y_pred, target_names=action_le.classes_, zero_division=0))

In [ ]:
gru_model.export(f"{artifacts_dir}/next_action_gru")
json.dump({int(i): cls for i,cls in enumerate(action_le.classes_)},
          open(f"{artifacts_dir}/next_action_label_map.json","w"), ensure_ascii=False, indent=2)
# !python -m tf2onnx.convert --saved-model artifacts/next_action_gru \
#   --output artifacts/next_action_gru.onnx --opset 13

#  ██████╗  █████╗ ██████╗ ████████╗      ██████╗
#  ██╔══██╗██╔══██╗██╔══██╗╚══██╔══╝     ██╔════╝
#  ██████╔╝███████║██████╔╝   ██║        ██║
#  ██╔═══╝ ██╔══██║██╔══██╗   ██║        ██║
#  ██║     ██║  ██║██║  ██║   ██║        ╚██████╗
#  ╚═╝     ╚═╝  ╚═╝╚═╝  ╚═╝   ╚═╝         ╚═════╝
#  XGBOOST — Daily trend forecasting

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

df_daily = df.copy()
df_daily["date"] = df_daily["createdAt"].dt.date
daily_counts = (df_daily.groupby(["date","action"]).size()
                .reset_index(name="count"))
daily_counts["date"]      = pd.to_datetime(daily_counts["date"])
daily_counts["dayofweek"] = daily_counts["date"].dt.dayofweek
daily_counts["month_num"] = daily_counts["date"].dt.month
daily_counts["day"]       = daily_counts["date"].dt.day
daily_counts["is_weekend"]= (daily_counts["dayofweek"]>=5).astype(int)

# Encode action
actl = LabelEncoder()
daily_counts["action_id"] = actl.fit_transform(daily_counts["action"])
daily_counts = daily_counts.sort_values(["action_id","date"]).reset_index(drop=True)

for lag in [1, 2, 3, 7, 14, 30]:
    daily_counts[f"lag_{lag}"] = daily_counts.groupby("action_id")["count"].shift(lag)

daily_counts["rolling_mean_7"] = daily_counts.groupby("action_id")["count"].shift(1).rolling(7).mean()
daily_counts["rolling_std_7"]  = daily_counts.groupby("action_id")["count"].shift(1).rolling(7).std()
daily_counts["rolling_mean_30"]= daily_counts.groupby("action_id")["count"].shift(1).rolling(30).mean()
daily_counts = daily_counts.dropna().reset_index(drop=True)

feature_cols_trend = ["action_id","dayofweek","month_num","day","is_weekend",
                      "lag_1","lag_2","lag_3","lag_7","lag_14","lag_30",
                      "rolling_mean_7","rolling_std_7","rolling_mean_30"]
X_trend  = daily_counts[feature_cols_trend]
y_trend  = daily_counts["count"]
split_tr = int(len(daily_counts)*0.8)
X_tr_t, X_te_t = X_trend.iloc[:split_tr], X_trend.iloc[split_tr:]
y_tr_t, y_te_t = y_trend.iloc[:split_tr], y_trend.iloc[split_tr:]

In [ ]:
trend_model = XGBRegressor(
    n_estimators=600, learning_rate=0.03, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    early_stopping_rounds=30, eval_metric="mae",
)
trend_model.fit(X_tr_t, y_tr_t, eval_set=[(X_te_t, y_te_t)], verbose=50)

y_pred_t = trend_model.predict(X_te_t)
mae  = mean_absolute_error(y_te_t, y_pred_t)
rmse = math.sqrt(mean_squared_error(y_te_t, y_pred_t))
print(f"XGBoost  MAE  : {mae:.3f}")
print(f"XGBoost  RMSE : {rmse:.3f}")

joblib.dump(trend_model, f"{artifacts_dir}/trend_xgboost.pkl")
json.dump(feature_cols_trend, open(f"{artifacts_dir}/trend_feature_cols.json","w"))

#  ██████╗  █████╗ ██████╗ ████████╗    ██████╗
#  ██╔══██╗██╔══██╗██╔══██╗╚══██╔══╝    ██╔══██╗
#  ██████╔╝███████║██████╔╝   ██║       ██║  ██║
#  ██╔═══╝ ██╔══██║██╔══██╗   ██║       ██║  ██║
#  ██║     ██║  ██║██║  ██║   ██║       ██████╔╝
#  ╚═╝     ╚═╝  ╚═╝╚═╝  ╚═╝   ╚═╝       ╚═════╝
#  ANOMALY TYPE CLASSIFIER
#
#  Answers: "WHICH TYPE of anomaly is this?"
#  Runs AFTER the LSTM AE has flagged a sequence as anomalous.
#  Trained on labeled anomalous sequences (is_anomaly=1, anomaly_type=X).
#  Produces one of 6 labels → fed to LLM for explanation.

In [ ]:
ANOMALY_TYPES = ["rapid_fire","unusual_hour","geo_jump",
                 "repeated_fail","skip_login","impossible_seq"]

# We train on ALL sequences but use anomaly_type as target.
# Normal sequences get label "normal" → the classifier also handles "is this
# actually normal?" which adds robustness when AE has false positives.
#
# Target: 7 classes (normal + 6 anomaly types)
atype_le = LabelEncoder()
y_atype_all_arr = np.array(y_atype_all)
y_type_encoded  = atype_le.fit_transform(y_atype_all_arr)
num_anomaly_classes = len(atype_le.classes_)
print("Anomaly type classes:", atype_le.classes_)
print("Class distribution:"); print(pd.Series(y_atype_all_arr).value_counts())

# ── Build train/test for type classifier ──
y_type_train = y_type_encoded[:split]
y_type_test  = y_type_encoded[split:]

# Class weights for type classifier
classes_atype = np.unique(y_type_train)
raw_w_atype = compute_class_weight("balanced", classes=classes_atype, y=y_type_train)
raw_w_atype = np.clip(raw_w_atype, 0.1, 15.0)
cw_atype = {int(c): float(w) for c,w in zip(classes_atype, raw_w_atype)}
for c in range(num_anomaly_classes):
    cw_atype.setdefault(c, 1.0)

In [ ]:
tc_in = layers.Input(shape=(SEQ_LEN, input_dim))
tx = layers.Masking(mask_value=0.0)(tc_in)
tx = layers.Bidirectional(layers.GRU(128, return_sequences=True))(tx)
tx = layers.Dropout(0.25)(tx)
tx = layers.Bidirectional(layers.GRU(64, return_sequences=False))(tx)
tx = layers.Dropout(0.25)(tx)
tx = layers.Dense(128, activation="relu")(tx)
tx = layers.Dropout(0.20)(tx)
tc_out = layers.Dense(num_anomaly_classes, activation="softmax")(tx)

type_classifier = models.Model(tc_in, tc_out, name="anomaly_type_classifier")
type_classifier.compile(
    optimizer=tf.keras.optimizers.Adam(5e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
type_classifier.summary()

In [ ]:
X_tr_tc, X_val_tc, y_tr_tc, y_val_tc = train_test_split(
    X_train, y_type_train, test_size=0.15, shuffle=False)

es_tc = callbacks.EarlyStopping(monitor="val_accuracy", patience=12,
                                 restore_best_weights=True, mode="max")
lr_tc = callbacks.ReduceLROnPlateau(monitor="val_accuracy", factor=0.5,
                                     patience=4, min_lr=1e-6, mode="max")

history_tc = type_classifier.fit(
    X_tr_tc, y_tr_tc,
    validation_data=(X_val_tc, y_val_tc),
    epochs=60, batch_size=128,
    class_weight=cw_atype,
    callbacks=[es_tc, lr_tc], verbose=1,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(history_tc.history["accuracy"], label="train")
ax1.plot(history_tc.history["val_accuracy"], label="val")
ax1.set_title("Type Classifier Accuracy"); ax1.legend()
ax2.plot(history_tc.history["loss"], label="train")
ax2.plot(history_tc.history["val_loss"], label="val")
ax2.set_title("Type Classifier Loss"); ax2.legend()
plt.tight_layout(); plt.show()

In [ ]:
y_tc_prob = type_classifier.predict(X_test, verbose=0)
y_tc_pred = np.argmax(y_tc_prob, axis=1)

print("Overall accuracy:", accuracy_score(y_type_test, y_tc_pred))
print()
print(classification_report(y_type_test, y_tc_pred,
      target_names=atype_le.classes_, zero_division=0))

# Confusion matrix
cm = confusion_matrix(y_type_test, y_tc_pred)
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay(cm, display_labels=atype_le.classes_).plot(ax=ax, colorbar=False)
plt.title("Anomaly Type Classifier — Confusion Matrix")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

# ── Evaluate only on TRULY anomalous test sequences ──
anomaly_idx = np.where(y_type_test != atype_le.transform(["normal"])[0])[0]
if len(anomaly_idx) > 0:
    y_true_anom = y_type_test[anomaly_idx]
    y_pred_anom = y_tc_pred[anomaly_idx]
    print(f"\nAmong {len(anomaly_idx)} truly anomalous sequences:")
    print(classification_report(y_true_anom, y_pred_anom,
          target_names=[c for c in atype_le.classes_ if c != "normal"],
          labels=[i for i,c in enumerate(atype_le.classes_) if c != "normal"],
          zero_division=0))

In [ ]:
type_classifier.export(f"{artifacts_dir}/anomaly_type_classifier")
json.dump({int(i): cls for i,cls in enumerate(atype_le.classes_)},
          open(f"{artifacts_dir}/anomaly_type_label_map.json","w"), ensure_ascii=False, indent=2)
# !python -m tf2onnx.convert --saved-model artifacts/anomaly_type_classifier \
#   --output artifacts/anomaly_type_classifier.onnx --opset 13

In [ ]:
# Upload anomaly_test_set.csv (generated by anomaly_test_generator.py)
# then run:

def evaluate_on_test_set(csv_path: str):
    """
    Load an externally generated test CSV (anomaly_test_generator.py output),
    run both the LSTM AE and the Type Classifier, print full evaluation.
    """
    df_t = pd.read_csv(csv_path)
    df_t["createdAt"] = pd.to_datetime(df_t["createdAt"], utc=True, errors="coerce")
    for c in ["action","type","subType","device","countryCode","city","status","userAgent","anomaly_type"]:
        if c in df_t.columns:
            df_t[c] = df_t[c].fillna("UNKNOWN").astype(str)
    if "is_anomaly" not in df_t.columns:
        df_t["is_anomaly"] = 0
    if "sessionLength" not in df_t.columns:
        df_t["sessionLength"] = df_t.groupby(["insuredId","sessionNumber"])["sequenceInSession"].transform("max")

    # Apply same encoders (handle unseen labels gracefully)
    def safe_transform(le, series):
        classes_set = set(le.classes_)
        return np.array([le.transform([v])[0] if v in classes_set else 0 for v in series])

    df_t["action_id"]  = safe_transform(action_le,  df_t["action"])
    df_t["device_id"]  = safe_transform(device_le,  df_t["device"])
    df_t["country_id"] = safe_transform(country_le, df_t["countryCode"])
    df_t["type_id"]    = safe_transform(type_le,    df_t["type"])
    df_t["subType_filled"] = df_t["subType"].fillna("NONE")
    df_t["subtype_id"] = safe_transform(subtype_le, df_t["subType_filled"])

    df_t["hour"]      = df_t["createdAt"].dt.hour
    df_t["dayofweek"] = df_t["createdAt"].dt.dayofweek
    df_t["month_num"] = df_t["createdAt"].dt.month
    df_t["hour_sin"]  = np.sin(2*np.pi*df_t["hour"]/24)
    df_t["hour_cos"]  = np.cos(2*np.pi*df_t["hour"]/24)
    df_t["dow_sin"]   = np.sin(2*np.pi*df_t["dayofweek"]/7)
    df_t["dow_cos"]   = np.cos(2*np.pi*df_t["dayofweek"]/7)
    df_t["is_ok"]     = (df_t["status"]=="OK").astype(int)
    df_t["is_ko"]     = (df_t["status"]=="KO").astype(int)

    SK = ["insuredId","sessionNumber"]
    df_t["prev_time"] = df_t.groupby(SK)["createdAt"].shift(1)
    df_t["delta_seconds"] = (df_t["createdAt"]-df_t["prev_time"]).dt.total_seconds().fillna(0)
    df_t["delta_scaled"]  = scaler_delta.transform(df_t[["delta_seconds".replace("_seconds","_seconds_clipped")
                                                           if "delta_seconds_clipped" in df_t.columns
                                                           else "delta_seconds"]].clip(0, 3600))
    df_t["seq_pos_norm"]     = df_t["sequenceInSession"] / df_t["sessionLength"]
    df_t["is_session_start"] = (df_t["sequenceInSession"]==1).astype(int)
    df_t["is_session_end"]   = (df_t["sequenceInSession"]==df_t["sessionLength"]).astype(int)
    df_t["prev_action_id"]   = (df_t.groupby(SK)["action_id"].shift(1).fillna(-1).astype(int)+1)
    df_t["ko_count_so_far"]  = (df_t.groupby(SK)["is_ko"].cumsum().shift(1).fillna(0).astype(int))
    df_t["session_len_norm"] = df_t["sessionLength"] / max_session_len
    df_t["ip_prev"]    = df_t.groupby(SK)["ip"].shift(1)
    df_t["ip_changed"] = ((df_t["ip"]!=df_t["ip_prev"]) & df_t["ip_prev"].notna()).astype(int)

    X_ext, y_anom_ext, y_type_ext, _, _ = build_sequences(df_t, SEQ_LEN)[:5]

    # AE evaluation
    preds_ae  = autoencoder.predict(X_ext, verbose=0)
    errors_ae = np.mean(np.square(X_ext - preds_ae), axis=(1,2))
    flags_ae  = (errors_ae > threshold).astype(int)

    print("═"*50)
    print("LSTM AE — Binary Detection")
    print(f"  Flagged {flags_ae.sum()}/{len(flags_ae)}  ({100*flags_ae.mean():.1f}%)")
    if y_anom_ext.sum() > 0:
        auc = roc_auc_score(y_anom_ext, errors_ae)
        print(f"  AUC-ROC : {auc:.4f}")

    # Type classifier evaluation
    preds_tc  = type_classifier.predict(X_ext, verbose=0)
    labels_tc = np.argmax(preds_tc, axis=1)
    safe_y_type = np.array([atype_le.transform([t])[0] if t in set(atype_le.classes_) else 0
                             for t in y_type_ext])
    print("\nAnomaly Type Classifier")
    print(classification_report(safe_y_type, labels_tc,
          target_names=atype_le.classes_, zero_division=0))

# Uncomment once you've uploaded anomaly_test_set.csv:
# evaluate_on_test_set("anomaly_test_set.csv")

In [ ]:
# The outputs from the 4 models flow into your LLM like this:

LLM_PROMPT_TEMPLATE = """
You are a security analyst for a health insurance self-service portal.

SESSION SUMMARY:
  User: {insured_id}
  Device: {device}
  Country: {country}
  Session start: {session_start}
  Session length: {session_length} actions

ACTION SEQUENCE (last {seq_len} steps):
{action_sequence}

MODEL OUTPUTS:
  LSTM Autoencoder reconstruction error : {ae_score:.4f}  (threshold: {ae_threshold:.4f})
  Anomaly detected                       : {is_anomaly}
  Anomaly type (classifier)              : {anomaly_type}
  Confidence                             : {confidence:.1%}

  Next-action prediction (GRU top-3)    : {next_actions}

TASK:
1. Explain in plain French what anomaly type "{anomaly_type}" means in this context.
2. Describe the risk level (low / medium / high) with justification.
3. Suggest one concrete action the security team should take.
"""

print("LLM prompt template ready.")
print("\nAll models saved to artifacts/:")
import os
for f in sorted(os.listdir("artifacts")):
    print(f"  {f}")